# Документ OCR pipeline (Google Colab)

Рабочий baseline для тестового задания ML Engineer:
1. Загружает фото документа
2. Выравнивает документ (perspective transform)
3. Распознаёт текст (EasyOCR, локально, без токенов)
4. Извлекает структурированные поля (ФИО, дата рождения, номер документа)
5. Сохраняет выровненное изображение, изображение с боксами и JSON


In [ ]:
!pip -q install opencv-python-headless easyocr matplotlib rapidfuzz pillow transformers accelerate sentencepiece


In [ ]:
import os
import re
import json
import unicodedata
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr
import torch
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
from transformers import pipeline


In [ ]:
# --- Геометрия: выравнивание документа ---
def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype='float32')

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped


def align_document(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(gray, 60, 180)

    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    doc_cnt = None
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            doc_cnt = approx.reshape(4, 2)
            break

    if doc_cnt is None:
        # fallback: если контур не найден, возвращаем исходное изображение
        return image_bgr

    aligned = four_point_transform(image_bgr, doc_cnt.astype('float32'))
    return aligned

In [ ]:
# --- OCR + визуализация ---
reader = easyocr.Reader(['ru', 'en'], gpu=torch.cuda.is_available())


def run_ocr(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = reader.readtext(rgb, detail=1, paragraph=False)
    return results


def load_font(size=20):
    font_candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
    ]
    for font_path in font_candidates:
        try:
            return ImageFont.truetype(font_path, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


def draw_boxes(image_bgr, ocr_results):
    # PIL нужен для корректной отрисовки русского текста (без ?)
    rgb = cv2.cvtColor(image_bgr.copy(), cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(rgb)
    draw = ImageDraw.Draw(image_pil)
    font = load_font(size=max(14, image_bgr.shape[1] // 60))

    for item in ocr_results:
        box, text, conf = item
        pts = [(int(p[0]), int(p[1])) for p in box]
        draw.line(pts + [pts[0]], fill=(0, 255, 0), width=3)

        x, y = pts[0]
        label = f'{text[:40]} ({float(conf):.2f})'
        draw.text((x, max(0, y - 24)), label, fill=(255, 0, 0), font=font)

    out_rgb = np.array(image_pil)
    return cv2.cvtColor(out_rgb, cv2.COLOR_RGB2BGR)

In [ ]:
# --- Извлечение полей из OCR текста (LLM обязательно) ---
date_pattern = re.compile(r'\b(\d{2}[./-]\d{2}[./-]\d{4})\b')
doc_number_pattern = re.compile(r'\b(\d{2}\s?\d{2}\s?\d{6}|\d{9,12}|\d{10})\b')
fio_pattern = re.compile(r'^[А-ЯЁ][А-ЯЁ-]+(?:\s+[А-ЯЁ][А-ЯЁ-]+){1,2}$')

stop_phrases = {
    'ВОДИТЕЛЬСКОЕ УДОСТОВЕРЕНИЕ', 'УДОСТОВЕРЕНИЕ ЛИЧНОСТИ', 'ПАСПОРТ',
    'РОССИЙСКАЯ ФЕДЕРАЦИЯ', 'DRIVING LICENCE', 'DRIVER LICENSE', 'IDENTITY CARD',
}

blocked_geo_tokens = {
    'ОБЛ', 'ОБЛАСТЬ', 'ГОРОД', 'Г', 'МОСКВА', 'РАЙОН', 'РЕСПУБЛИКА', 'КРАЙ',
    'АВТ', 'ФЕДЕРАЦИЯ', 'РФ', 'RUS'
}

latin_to_cyr = str.maketrans({
    'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'H': 'Н', 'K': 'К', 'M': 'М',
    'O': 'О', 'P': 'Р', 'T': 'Т', 'X': 'Х', 'Y': 'У'
})

# LLM extractor обязателен
LLM_MODEL_NAME = os.getenv('DOC_LLM_MODEL', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0')
_llm_pipe = None


def clean_text(s):
    return re.sub(r'\s+', ' ', str(s).strip())


def normalize_ocr_text(s):
    s = clean_text(s).upper()
    s = unicodedata.normalize('NFKC', s)
    s = s.translate(latin_to_cyr)
    s = re.sub(r'[^А-ЯЁ\-\s0-9./]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def strip_to_name_chars(s):
    return re.sub(r'[^А-ЯЁ\-\s]', '', s).strip()


def extract_line_items(ocr_results):
    items = []
    for box, text, conf in ocr_results:
        if float(conf) < 0.20:
            continue
        norm = normalize_ocr_text(text)
        if not norm:
            continue
        ys = [p[1] for p in box]
        xs = [p[0] for p in box]
        y_center = float(sum(ys) / len(ys))
        x_left = float(min(xs))
        items.append({
            'raw': str(text),
            'norm': norm,
            'conf': float(conf),
            'y_center': y_center,
            'x_left': x_left
        })
    items.sort(key=lambda d: (d['y_center'], d['x_left']))
    return items


def extract_fields_heuristic(ocr_results):
    # fallback на случай, если LLM не вернет валидный JSON
    line_items = extract_line_items(ocr_results)
    lines = [x['norm'] for x in line_items]
    joined = ' '.join(lines)

    birth_date = None
    date_match = date_pattern.search(joined)
    if date_match:
        birth_date = date_match.group(1).replace('-', '.').replace('/', '.')

    doc_number = None
    normalized_joined = joined.replace('O', '0').replace('О', '0')
    num_match = doc_number_pattern.search(normalized_joined)
    if num_match:
        doc_number = re.sub(r'\s+', ' ', num_match.group(1)).strip()

    fio = None
    for line in lines:
        s = strip_to_name_chars(line)
        if not s or s in stop_phrases:
            continue
        words = s.split()
        if any(w in blocked_geo_tokens for w in words):
            continue
        if fio_pattern.match(s):
            fio = s
            break

    return {
        'full_name': fio,
        'birth_date': birth_date,
        'document_number': doc_number,
        'ocr_lines_normalized': lines
    }


def get_llm_pipe():
    global _llm_pipe
    if _llm_pipe is not None:
        return _llm_pipe
    _llm_pipe = pipeline(
        'text-generation',
        model=LLM_MODEL_NAME,
        device_map='auto',
        torch_dtype='auto'
    )
    return _llm_pipe


def parse_json_fragment(text):
    m = re.search(r'\{.*\}', text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def llm_extract_fields(ocr_lines):
    pipe = get_llm_pipe()
    prompt = f"""Из OCR строк документа извлеки поля и верни строго JSON.
Ключи: full_name, birth_date, document_number.
Формат даты: DD.MM.YYYY. Если поля нет — null.
OCR lines:
{json.dumps(ocr_lines, ensure_ascii=False)}
JSON:"""

    out = pipe(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )
    generated = out[0]['generated_text']
    parsed = parse_json_fragment(generated)
    if parsed is None:
        raise ValueError(f'LLM не вернула валидный JSON. Ответ: {generated[:400]}')

    return {
        'full_name': parsed.get('full_name'),
        'birth_date': parsed.get('birth_date'),
        'document_number': parsed.get('document_number')
    }


def extract_fields(ocr_results):
    # 1) OCR -> нормализованные строки
    heuristic = extract_fields_heuristic(ocr_results)

    # 2) Обязательный этап LLM
    llm_fields = None
    llm_error = None
    try:
        llm_fields = llm_extract_fields(heuristic.get('ocr_lines_normalized', []))
    except Exception as e:
        llm_error = str(e)

    # 3) Возвращаем LLM как primary; fallback на heuristic по пустым полям
    final_fields = {
        'full_name': None,
        'birth_date': None,
        'document_number': None,
        'ocr_lines_normalized': heuristic.get('ocr_lines_normalized', []),
        'llm_used': True,
        'llm_model_name': LLM_MODEL_NAME,
        'llm_fields': llm_fields,
        'llm_error': llm_error,
    }

    for key in ('full_name', 'birth_date', 'document_number'):
        if llm_fields and llm_fields.get(key):
            final_fields[key] = llm_fields[key]
        else:
            final_fields[key] = heuristic.get(key)

    return final_fields


In [ ]:
# --- Основной пайплайн ---
def to_python(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, list):
        return [to_python(x) for x in obj]
    if isinstance(obj, tuple):
        return [to_python(x) for x in obj]
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    return obj


def process_document_image(image_path, out_dir='outputs'):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f'Не удалось прочитать изображение: {image_path}')

    aligned = align_document(image_bgr)
    ocr_results = run_ocr(aligned)
    annotated = draw_boxes(aligned, ocr_results)
    fields = extract_fields(ocr_results)

    stem = Path(image_path).stem
    aligned_path = out_dir / f'{stem}_aligned.jpg'
    annotated_path = out_dir / f'{stem}_annotated.jpg'
    json_path = out_dir / f'{stem}_fields.json'

    cv2.imwrite(str(aligned_path), aligned)
    cv2.imwrite(str(annotated_path), annotated)

    payload = {
        'input_image': str(image_path),
        'aligned_image': str(aligned_path),
        'annotated_image': str(annotated_path),
        'extractor_config': {
            'llm_required': True,
            'llm_model_name': LLM_MODEL_NAME
        },
        'fields': fields,
        'ocr': [
            {
                'box': to_python(item[0]),
                'text_raw': str(item[1]),
                'text_normalized': normalize_ocr_text(item[1]),
                'confidence': float(item[2])
            } for item in ocr_results
        ]
    }

    payload = to_python(payload)

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    with open(str(json_path).replace('.json', '_utf8sig.json'), 'w', encoding='utf-8-sig') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return payload


In [ ]:
# --- Smoke tests (LLM обязателен + fallback) ---
sample_ocr = [
    ([[0,0],[100,0],[100,10],[0,10]], 'ВОДИТЕЛЬСКОЕ УДОСТОВЕРЕНИЕ', 0.99),
    ([[0,12],[100,12],[100,22],[0,22]], 'СЕРГЕЙ', 0.95),
    ([[0,24],[100,24],[100,34],[0,34]], 'Е Е', 0.95),
    ([[0,36],[100,36],[100,46],[0,46]], '22.05.1955', 0.95),
    ([[0,48],[100,48],[100,58],[0,58]], '77 07 123456', 0.98),
]

# 1) Эвристика как fallback
h = extract_fields_heuristic(sample_ocr)
assert h['document_number'] == '77 07 123456'
assert h['birth_date'] == '22.05.1955'

# 2) Полный extract_fields должен отмечать llm_used=True
f = extract_fields(sample_ocr)
assert f['llm_used'] is True
assert f['document_number'] is not None
print('Smoke test passed:', f)

tmp_payload = {'fields': f, 'ocr': [{'box': sample_ocr[0][0], 'confidence': sample_ocr[0][2]}]}
json.dumps(to_python(tmp_payload), ensure_ascii=False)
print('JSON serialization smoke test passed')


In [ ]:
# --- Загрузка файла в Colab и запуск ---
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))

result = process_document_image(image_name, out_dir='outputs')
print(json.dumps(result['fields'], ensure_ascii=False, indent=2))

aligned_rgb = cv2.cvtColor(cv2.imread(result['aligned_image']), cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(cv2.imread(result['annotated_image']), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.title('Aligned')
plt.imshow(aligned_rgb)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Detections + OCR')
plt.imshow(annotated_rgb)
plt.axis('off')
plt.show()

In [ ]:
# --- Скачать артефакты на локальную машину ---
files.download(result['aligned_image'])
files.download(result['annotated_image'])
files.download(f"outputs/{Path(image_name).stem}_fields.json")